In [12]:
!pip install scikeras

In [17]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [1]:
!pip install scikit-learn==1.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 77.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.9.0
    Uninstalling scikit-learn-1.9.0:
      Successfully uninstalled scikit-learn-1.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data prepared: X_train_scaled and y_train are ready.")

Data prepared: X_train_scaled and y_train are ready.


In [3]:
def create_model(optimizer='adam', learning_rate=0.001, activation='relu', neurons=32, num_layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation=activation, input_shape=(X_train_scaled.shape[1],)))
    for _ in range(num_layers - 1):
        model.add(Dense(neurons, activation=activation))
    model.add(Dense(1, activation='sigmoid'))

    opt = Adam(learning_rate=learning_rate) if optimizer == 'adam' else tf.keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("create_model function defined.")

create_model function defined.


 Manual Search CV


In [4]:
manual_params = [
    {'lr': 0.001, 'act': 'relu', 'n': 32, 'l': 1, 'e': 10, 'bs': 32},
    {'lr': 0.01, 'act': 'relu', 'n': 64, 'l': 2, 'e': 10, 'bs': 64},
    {'lr': 0.0005, 'act': 'tanh', 'n': 16, 'l': 1, 'e': 15, 'bs': 32}
]
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
results_manual = []

for i, params in enumerate(manual_params):
    fold_accuracies = []
    for train_idx, val_idx in kfold.split(X_train_scaled, y_train):
        model = create_model(learning_rate=params['lr'], activation=params['act'], neurons=params['n'], num_layers=params['l'])
        model.fit(X_train_scaled[train_idx], y_train[train_idx], epochs=params['e'], batch_size=params['bs'], verbose=0)
        _, accuracy = model.evaluate(X_train_scaled[val_idx], y_train[val_idx], verbose=0)
        fold_accuracies.append(accuracy)
    results_manual.append({'params': params, 'avg_accuracy': np.mean(fold_accuracies)})

best_manual_result = max(results_manual, key=lambda x: x['avg_accuracy'])
print(f"\nBest Manual Search Combination: {best_manual_result['params']}")
print(f"Best Manual Search Average Accuracy: {best_manual_result['avg_accuracy']:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Best Manual Search Combination: {'lr': 0.01, 'act': 'relu', 'n': 64, 'l': 2, 'e': 10, 'bs': 64}
Best Manual Search Average Accuracy: 0.9128


Randomized Search CV

In [5]:
from sklearn.model_selection import RandomizedSearchCV

model_keras_wrapper = KerasClassifier(model=create_model, verbose=0)
param_dist_random = {
    'model__learning_rate': [0.0001, 0.001, 0.01],
    'model__activation': ['relu', 'tanh'],
    'model__neurons': [32, 64],
    'model__num_layers': [1, 2],
    'batch_size': [32, 64],
    'epochs': [10, 20]
}

random_search = RandomizedSearchCV(estimator=model_keras_wrapper,
                                   param_distributions=param_dist_random,
                                   n_iter=10,
                                   cv=3,
                                   scoring='accuracy',
                                   verbose=0,
                                   random_state=42,
                                   n_jobs=-1)

print("Starting Randomized Search CV (this may take a moment)...")
random_search_result = random_search.fit(X_train_scaled, y_train)

print("\nBest Parameters for Randomized Search CV:", random_search_result.best_params_)
print("Best Accuracy for Randomized Search CV: %.4f" % random_search_result.best_score_)

Starting Randomized Search CV (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Best Parameters for Randomized Search CV: {'model__num_layers': 1, 'model__neurons': 64, 'model__learning_rate': 0.01, 'model__activation': 'relu', 'epochs': 10, 'batch_size': 32}
Best Accuracy for Randomized Search CV: 0.9228


 Grid Search CV


In [6]:
from sklearn.model_selection import GridSearchCV

param_grid_grid = {
    'model__learning_rate': [0.001, 0.01],
    'model__activation': ['relu'],
    'model__neurons': [32, 64],
    'model__num_layers': [1],
    'batch_size': [32, 64],
    'epochs': [10]
}

grid_search = GridSearchCV(estimator=model_keras_wrapper,
                           param_grid=param_grid_grid,
                           cv=3,
                           scoring='accuracy',
                           verbose=0,
                           n_jobs=-1)

print("Starting Grid Search CV (this may take a moment)...")
grid_search_result = grid_search.fit(X_train_scaled, y_train)

print("\nBest Parameters for Grid Search CV:", grid_search_result.best_params_)
print("Best Accuracy for Grid Search CV: %.4f" % grid_search_result.best_score_)

Starting Grid Search CV (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Best Parameters for Grid Search CV: {'batch_size': 64, 'epochs': 10, 'model__activation': 'relu', 'model__learning_rate': 0.01, 'model__neurons': 64, 'model__num_layers': 1}
Best Accuracy for Grid Search CV: 0.9200


In [7]:
print("--- HPO Results Summary ---")
print(f"Manual Search CV Best Accuracy: {best_manual_result['avg_accuracy']:.4f}")

if 'random_search_result' in locals():
    print(f"Randomized Search CV Best Accuracy: {random_search_result.best_score_:.4f}")
else:
    print("Randomized Search CV results not available (due to previous error).")

if 'grid_search_result' in locals():
    print(f"Grid Search CV Best Accuracy: {grid_search_result.best_score_:.4f}")
else:
    print("Grid Search CV results not available (due to previous error).")

print("\nNote: The 'best' parameters and accuracies might vary based on the specific search space and random states.")

--- HPO Results Summary ---
Manual Search CV Best Accuracy: 0.9128
Randomized Search CV Best Accuracy: 0.9228
Grid Search CV Best Accuracy: 0.9200

Note: The 'best' parameters and accuracies might vary based on the specific search space and random states.
